# 04 - Feature Engineering (Literature-Based)

Extracts features per field-year, 100% from published literature:
- 40 phenological metrics (10 per VI x 4 VIs: NDVI, EVI, GCVI, NDRE)
  - includes Left Shoulder (L), the curvature-based feature linked to flag leaf emergence
- 4 double logistic curve parameters (GCVI)
- 10 environmental features (latitude, longitude, photoperiod, GDD_M1, **GDD_M2**, **VD**, **fV** (Streck), **GDD_effective**, PPT_at_SOS, PPT_greenup)

**Thermal synthesis** (vernalization-aware):
- McMaster & Wilhelm 1997 → GDD Method 2
- Wang & Engel 1998 → β-function temp response for VD
- Streck 2003 → generalized sigmoid f(V) = VD⁵/(22.5⁵+VD⁵)

**References:** TIMESAT (Jonsson & Eklundh 2004), phenoC++ (Ruan 2023), Beck et al. (2006),
Zeng et al. (2020), Bloomfield et al. (2022), **Zhao et al. (2025) — Left Shoulder feature for flag leaf**

In [ ]:
import pandas as pd
import numpy as np
from scipy.signal import savgol_filter
from scipy.optimize import curve_fit
import os
import warnings
warnings.filterwarnings('ignore')

# === PATHS ===
OUTPUT_DIR = 'data/processed/features'
os.makedirs(OUTPUT_DIR, exist_ok=True)

APPROACHES = {
    'buffer': 'data/processed/buffer/hls_phenology_merged.parquet',
}

# Temperature data path (Daymet 1km)
TEMP_PATH_COMBINED = 'data/raw/satellite/daymet_daily_weather.csv'
TEMP_PATH_OLD = 'data/raw/satellite/daymet_daily_temperature.csv'
TEMP_PATH = TEMP_PATH_COMBINED if os.path.exists(TEMP_PATH_COMBINED) else TEMP_PATH_OLD

VIS = ['NDVI', 'EVI', 'GCVI', 'NDRE']

## 1. Load & Clean Data

In [ ]:
def load_and_clean(path):
    df = pd.read_parquet(path)
    for col in ['NDVI', 'EVI']:
        if col in df.columns:
            df.loc[df[col].abs() > 1, col] = np.nan
    for col in ['GCVI', 'CIre']:
        if col in df.columns:
            df.loc[(df[col] < -1) | (df[col] > 20), col] = np.nan
    if 'MTCI' in df.columns:
        df.loc[(df['MTCI'] < -5) | (df['MTCI'] > 15), 'MTCI'] = np.nan
    if 'NDRE' in df.columns:
        df.loc[df['NDRE'].abs() > 1, 'NDRE'] = np.nan
    return df

datasets = {}
for name, path in APPROACHES.items():
    datasets[name] = load_and_clean(path)
    df = datasets[name]
    has_flag = df['flag_emerging_doy'].notna()
    n_fy = df[has_flag].groupby(['field_id', 'year']).ngroups
    print(f'{name}: {len(df):,} rows, {df["field_id"].nunique()} fields, {n_fy} field-years with flag leaf')

## 2. Smoothing & Curve Fitting Functions

In [ ]:
def smooth_vi(doys, values):
    """Interpolate to daily + Savitzky-Golay smoothing."""
    if len(doys) < 5:
        return None, None
    idx = np.argsort(doys)
    doys_s, vals_s = np.array(doys)[idx], np.array(values)[idx]
    u_doys = np.unique(doys_s)
    u_vals = np.array([vals_s[doys_s == d].mean() for d in u_doys])
    target = np.arange(1, 366)
    daily = np.interp(target, u_doys, u_vals)
    win = min(15, len(daily))
    if win % 2 == 0: win -= 1
    if win < 4: return target, daily
    return target, savgol_filter(daily, win, 2)


def beck_dlogistic(t, c1, c2, c3, c4, c5, c6):
    """Beck double logistic function."""
    return c1 + c2 * (1.0/(1.0 + np.exp(-c3*(t - c4))) - 1.0/(1.0 + np.exp(-c5*(t - c6))))


def fit_double_logistic(doys, smoothed):
    """
    Improved double logistic fit:
    1. Multiple initial guesses
    2. Wider bounds
    3. Fallback: single logistic for ascending limb
    """
    vi_min = smoothed[:60].min()
    vi_max = smoothed[59:200].max()
    amp = max(vi_max - vi_min, 0.01)
    pos_doy = np.argmax(smoothed[59:200]) + 60

    best_popt = None
    best_r2 = -999

    # Strategy 1: Full double logistic with multiple initial guesses
    initial_guesses = [
        [vi_min, amp, 0.08, 90, 0.08, 180],
        [vi_min, amp, 0.10, 100, 0.10, 200],
        [vi_min, amp, 0.15, 110, 0.05, 190],
        [vi_min, amp, 0.05, 85, 0.12, 210],
        [vi_min, amp, 0.12, 105, 0.08, 175],
        [vi_min, amp*0.8, 0.10, 95, 0.10, 195],
        [vi_min, amp*1.2, 0.08, 100, 0.06, 220],
    ]

    bounds_lower = [vi_min - abs(amp), 0, 0.005, 30, 0.005, 120]
    bounds_upper = [vi_min + abs(amp), 5*amp, 2.0, 170, 2.0, 320]

    for p0 in initial_guesses:
        try:
            popt, _ = curve_fit(beck_dlogistic, doys, smoothed, p0=p0,
                               bounds=(bounds_lower, bounds_upper), maxfev=10000)
            predicted = beck_dlogistic(doys, *popt)
            ss_res = np.sum((smoothed - predicted) ** 2)
            ss_tot = np.sum((smoothed - smoothed.mean()) ** 2)
            r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0
            if popt[3] < popt[5] and r2 > best_r2:
                best_r2 = r2
                best_popt = popt
        except:
            continue

    # Strategy 2: Single logistic fallback (ascending limb only)
    if best_r2 < 0.5:
        try:
            ascending = smoothed[:pos_doy+30]
            ascending_doys = doys[:pos_doy+30]

            def single_logistic(t, base, a, k, t0):
                return base + a / (1.0 + np.exp(-k * (t - t0)))

            p0_s = [vi_min, amp, 0.1, 100]
            bounds_s = ([vi_min - abs(amp), 0, 0.005, 30],
                       [vi_min + abs(amp), 5*amp, 2.0, 170])
            popt_s, _ = curve_fit(single_logistic, ascending_doys, ascending,
                                  p0=p0_s, bounds=bounds_s, maxfev=10000)
            predicted_s = single_logistic(ascending_doys, *popt_s)
            ss_res_s = np.sum((ascending - predicted_s) ** 2)
            ss_tot_s = np.sum((ascending - ascending.mean()) ** 2)
            r2_s = 1 - ss_res_s / ss_tot_s if ss_tot_s > 0 else 0
            if r2_s > 0.5:
                best_popt = np.array([popt_s[0], popt_s[1], popt_s[2], popt_s[3],
                                      0.1, max(pos_doy + 30, popt_s[3] + 60)])
                best_r2 = r2_s
        except:
            pass

    if best_r2 > 0.5 and best_popt is not None:
        return best_popt
    return None


def photoperiod(lat, doy):
    """Day length (hours) from latitude and DOY."""
    lat_rad = np.radians(lat)
    decl = 23.45 * np.sin(np.radians(360.0/365.0 * (doy - 81)))
    decl_rad = np.radians(decl)
    cos_ha = -np.tan(lat_rad) * np.tan(decl_rad)
    cos_ha = np.clip(cos_ha, -1, 1)
    ha = np.degrees(np.arccos(cos_ha))
    return 2.0 * ha / 15.0

print('Functions defined (improved DL fitting).')

## 3. Load Temperature Data for GDD

In [ ]:
# Load temperature data (Daymet or ERA5)
gdd_data = None
if os.path.exists(TEMP_PATH):
    df_temp = pd.read_csv(TEMP_PATH)
    df_temp['date'] = pd.to_datetime(df_temp['date'], format='mixed')
    df_temp['doy'] = df_temp['date'].dt.dayofyear
    df_temp['year'] = df_temp['date'].dt.year
    
    # Handle different column names (Daymet: tmin/tmax, ERA5: Tmin/Tmax)
    if 'tmin' in df_temp.columns:
        df_temp = df_temp.rename(columns={'tmin': 'Tmin', 'tmax': 'Tmax'})
    
    # Ensure FIELDID is string
    if 'FIELDID' in df_temp.columns:
        df_temp['FIELDID'] = df_temp['FIELDID'].astype(str)
    
    # === Method 1 GDD (McMaster & Wilhelm 1997) — classic baseline ===
    df_temp['gdd_daily'] = np.maximum(0, (df_temp['Tmin'] + df_temp['Tmax']) / 2.0)
    
    # === Method 2 GDD (McMaster & Wilhelm 1997) — better for overwintering ===
    T_base, T_upper = 0.0, 35.0
    tmin_clip = df_temp['Tmin'].clip(lower=T_base, upper=T_upper)
    tmax_clip = df_temp['Tmax'].clip(lower=T_base, upper=T_upper)
    df_temp['gdd_m2_daily'] = ((tmax_clip + tmin_clip) / 2.0 - T_base).clip(lower=0)
    
    # === Daily Vernalization Days (Wang & Engel 1998 β-function, Streck cardinals Porter & Gawith 1999) ===
    df_temp['T_mean'] = (df_temp['Tmin'] + df_temp['Tmax']) / 2.0
    Tmin_vn, Topt_vn, Tmax_vn = 1.3, 4.9, 15.7
    # Wang & Engel Eq. 6: beta function temperature response
    alpha = np.log(2) / np.log((Tmax_vn - Tmin_vn) / (Topt_vn - Tmin_vn))
    T = df_temp['T_mean'].values
    f_vn = np.zeros_like(T)
    mask = (T > Tmin_vn) & (T < Tmax_vn)
    x = (T[mask] - Tmin_vn)
    xopt = (Topt_vn - Tmin_vn)
    f_vn[mask] = (2 * x**alpha * xopt**alpha - x**(2*alpha)) / (xopt**(2*alpha))
    f_vn = np.clip(f_vn, 0, 1)
    df_temp['vd_daily'] = f_vn
    
    # === Cumulative from Jan 1 ===
    df_temp = df_temp.sort_values(['FIELDID', 'date'])
    df_temp['gdd_cum']     = df_temp.groupby(['FIELDID', 'year'])['gdd_daily'].cumsum()       # M1
    df_temp['gdd_m2_cum']  = df_temp.groupby(['FIELDID', 'year'])['gdd_m2_daily'].cumsum()    # M2
    df_temp['vd_cum']      = df_temp.groupby(['FIELDID', 'year'])['vd_daily'].cumsum()         # PVDs
    
    # === Streck 2003 Eq. 2: generalized vernalization factor f(V) ===
    df_temp['fV'] = df_temp['vd_cum']**5 / (22.5**5 + df_temp['vd_cum']**5)
    
    # === Effective GDD (Method 2 × fV) ===
    df_temp['gdd_eff_daily'] = df_temp['gdd_m2_daily'] * df_temp['fV']
    df_temp['gdd_eff_cum']   = df_temp.groupby(['FIELDID', 'year'])['gdd_eff_daily'].cumsum()
    
    # === Precipitation ===
    if 'prcp' in df_temp.columns:
        df_temp['prcp_cum'] = df_temp.groupby(['FIELDID', 'year'])['prcp'].cumsum()
        print(f'  Precipitation loaded: cum PPT range 0 - {df_temp["prcp_cum"].max():.0f} mm')
    
    print(f'  Synthesis thermal features:')
    print(f'    GDD_M1 (classic): 0-{df_temp["gdd_cum"].max():.0f}')
    print(f'    GDD_M2 (McMaster method 2): 0-{df_temp["gdd_m2_cum"].max():.0f}')
    print(f'    VD cumulative (Wang-Engel+Streck): 0-{df_temp["vd_cum"].max():.1f}')
    print(f'    fV (Streck 2003): {df_temp["fV"].min():.3f}-{df_temp["fV"].max():.3f}')
    print(f'    GDD_effective (M2 × fV): 0-{df_temp["gdd_eff_cum"].max():.0f}')
    
    gdd_data = df_temp
    n_fields = df_temp['FIELDID'].nunique()
    print(f'Temperature data loaded: {len(df_temp):,} rows, {n_fields} fields')
    print(f'Date range: {df_temp["date"].min().date()} to {df_temp["date"].max().date()}')
    print(f'FIELDID sample: {df_temp["FIELDID"].iloc[:3].tolist()}')
    print(f'GDD range: {df_temp["gdd_cum"].min():.0f} to {df_temp["gdd_cum"].max():.0f}')
else:
    print(f'Temperature data not found at {TEMP_PATH}')
    print('Run 02c_daymet_download.py first.')
    print('GDD features will be NaN for now.')

## 4. Extract Features

In [ ]:
def extract_phenological_metrics(doys, smoothed, vi_name):
    """Extract 9 phenological metrics for one VI, one field-year."""
    feat = {}
    base_vi = smoothed[:60].min()
    feat[f'{vi_name}_base'] = base_vi
    peak_vi = smoothed[59:200].max()
    feat[f'{vi_name}_peak'] = peak_vi
    amplitude = peak_vi - base_vi
    feat[f'{vi_name}_amplitude'] = amplitude
    pos_doy = np.argmax(smoothed[59:200]) + 60
    feat[f'{vi_name}_POS'] = pos_doy
    deriv1 = np.gradient(smoothed, doys)
    spring_d1 = deriv1[59:pos_doy]
    feat[f'{vi_name}_greenup_rate'] = spring_d1.max() if len(spring_d1) > 0 else np.nan
    deriv2 = np.gradient(deriv1, doys)
    deriv3 = np.gradient(deriv2, doys)
    spring_d3 = deriv3[29:pos_doy]
    spring_doys = doys[29:pos_doy]
    feat[f'{vi_name}_SOS'] = spring_doys[np.argmax(spring_d3)] if len(spring_d3) > 0 else np.nan
    spring_d2 = deriv2[59:pos_doy]
    spring_doys_d2 = doys[59:pos_doy]
    midpoint = np.nan
    if len(spring_d2) > 1:
        for k in range(1, len(spring_d2)):
            if spring_d2[k-1] > 0 and spring_d2[k] <= 0:
                frac = spring_d2[k-1] / (spring_d2[k-1] - spring_d2[k])
                midpoint = spring_doys_d2[k-1] + frac
                break
    feat[f'{vi_name}_greenup_midpoint'] = midpoint
    sos = feat[f'{vi_name}_SOS']
    feat[f'{vi_name}_duration_greenup'] = pos_doy - sos if not np.isnan(sos) else np.nan
    if not np.isnan(sos):
        sos_int = max(0, int(sos) - 1)
        pos_int = min(364, pos_doy - 1)
        feat[f'{vi_name}_integrated'] = np.trapz(smoothed[sos_int:pos_int+1])
    else:
        feat[f'{vi_name}_integrated'] = np.nan
    # Left Shoulder (L): DOY of maximum curvature on the up-slope [SOS, POS]
    # K(t) = |f''(t)| / (1 + f'(t)^2)^1.5  (Zhao et al. 2025, in silico Plants)
    if not np.isnan(sos) and (pos_doy - int(sos)) > 5:
        sos_i = max(0, int(sos) - 1)
        pos_i = min(len(doys) - 1, pos_doy - 1)
        K = np.abs(deriv2[sos_i:pos_i+1]) / (1 + deriv1[sos_i:pos_i+1]**2)**1.5
        feat[f'{vi_name}_LeftShoulder'] = doys[sos_i + np.argmax(K)] if len(K) > 0 else np.nan
    else:
        feat[f'{vi_name}_LeftShoulder'] = np.nan
    return feat


def extract_all_features(df, approach_name, gdd_data=None):
    """Extract all features for all field-years."""
    has_flag = df[df['flag_emerging_doy'].notna()]
    field_years = has_flag.groupby(['field_id', 'year']).first().reset_index()
    
    print(f'\n{approach_name}: Processing {len(field_years)} field-years...')
    
    csb_to_buf = None
    if gdd_data is not None:
        mapping_path = 'data/processed/csb_to_buf_mapping.csv'
        if os.path.exists(mapping_path):
            csb_to_buf = pd.read_csv(mapping_path)
            csb_to_buf['field_id'] = csb_to_buf['field_id'].astype(str)
            csb_to_buf = dict(zip(csb_to_buf['field_id'], csb_to_buf['nearest_BUF']))
            print(f'  Loaded CSB->BUF mapping ({len(csb_to_buf)} entries)')
    
    records = []
    n_dl_success = 0
    n_gdd_success = 0
    
    for idx, fy in field_years.iterrows():
        fid, year = fy['field_id'], fy['year']
        sub = df[(df['field_id'] == fid) & (df['year'] == year)]
        
        if len(sub) < 5:
            continue
        
        row = {
            'field_id': fid, 'year': year,
            'flag_true_doy': fy['flag_emerging_doy'], 'n_obs': len(sub),
        }
        
        # --- A. Phenological metrics per VI (9 x 4 = 36 features) ---
        ndvi_smoothed = None
        ndvi_doys = None
        for vi in VIS:
            if vi not in sub.columns:
                continue
            valid = sub[sub[vi].notna()]
            if len(valid) < 5:
                continue
            doys, smoothed = smooth_vi(valid['doy'].values, valid[vi].values)
            if doys is None:
                continue
            metrics = extract_phenological_metrics(doys, smoothed, vi)
            row.update(metrics)
            if vi == 'NDVI':
                ndvi_smoothed = smoothed
                ndvi_doys = doys
            if vi == 'GCVI':
                gcvi_smoothed = smoothed
                gcvi_doys = doys
        
        # --- B. Double logistic parameters (GCVI, 4 features) ---
        gcvi_smoothed = locals().get('gcvi_smoothed', None)
        gcvi_doys = locals().get('gcvi_doys', None)
        if gcvi_smoothed is not None:
            dl_params = fit_double_logistic(gcvi_doys, gcvi_smoothed)
            if dl_params is not None:
                row['DL_c3_greenup_steepness'] = dl_params[2]
                row['DL_c4_greenup_midpoint'] = dl_params[3]
                row['DL_c5_senesc_steepness'] = dl_params[4]
                row['DL_c6_senesc_midpoint'] = dl_params[5]
                n_dl_success += 1
        
        # --- C. Environmental features (5 features) ---
        lat = fy.get('lat', np.nan)
        lon = fy.get('lon', np.nan)
        row['latitude'] = lat
        row['longitude'] = lon
        
        sos = row.get('NDVI_SOS', np.nan)
        pos = row.get('NDVI_POS', np.nan)
        if not np.isnan(lat) and not np.isnan(sos):
            row['photoperiod_at_SOS'] = photoperiod(lat, sos)
        else:
            row['photoperiod_at_SOS'] = np.nan
        
        # Synthesis thermal features (McMaster 1997 + Wang-Engel 1998 + Streck 2003)
        for k in ['GDD_at_SOS','GDD_M2_at_SOS','VD_at_SOS','fV_at_SOS','GDD_eff_at_SOS',
                  'PPT_at_SOS','PPT_greenup']:
            row[k] = np.nan
        
        if gdd_data is not None and not np.isnan(sos):
            gdd_fid = fid
            if csb_to_buf is not None and fid in csb_to_buf:
                gdd_fid = csb_to_buf[fid]
            field_gdd = gdd_data[(gdd_data['FIELDID'] == gdd_fid) & (gdd_data['year'] == year)]
            if len(field_gdd) > 0:
                at_sos = field_gdd[field_gdd['doy'] <= int(sos)]
                # Classic GDD (Method 1)
                row['GDD_at_SOS'] = at_sos['gdd_cum'].max() if len(at_sos) > 0 else np.nan
                if not np.isnan(row['GDD_at_SOS']): n_gdd_success += 1
                # Method 2 GDD (McMaster & Wilhelm 1997)
                if 'gdd_m2_cum' in field_gdd.columns:
                    row['GDD_M2_at_SOS'] = at_sos['gdd_m2_cum'].max()
                # Vernalization days (Wang-Engel β + Streck cardinals)
                if 'vd_cum' in field_gdd.columns:
                    row['VD_at_SOS'] = at_sos['vd_cum'].max()
                # Streck 2003 f(V) sigmoid
                if 'fV' in field_gdd.columns:
                    row['fV_at_SOS'] = at_sos['fV'].max()
                # Effective GDD (M2 × fV)
                if 'gdd_eff_cum' in field_gdd.columns:
                    row['GDD_eff_at_SOS'] = at_sos['gdd_eff_cum'].max()
                # Precipitation
                if 'prcp_cum' in field_gdd.columns:
                    row['PPT_at_SOS'] = at_sos['prcp_cum'].max()
                    if not np.isnan(pos):
                        sub_ppt = field_gdd[(field_gdd['doy'] > int(sos)) & (field_gdd['doy'] <= int(pos))]
                        row['PPT_greenup'] = sub_ppt['prcp'].sum() if len(sub_ppt) > 0 else np.nan
        
        records.append(row)
        
        if (idx + 1) % 500 == 0:
            print(f'  {idx+1}/{len(field_years)} processed...')
    
    df_feat = pd.DataFrame(records)
    print(f'  Done: {len(df_feat)} field-years')
    print(f'  DL fit success: {n_dl_success} ({n_dl_success/len(df_feat)*100:.0f}%)')
    print(f'  GDD success: {n_gdd_success} ({n_gdd_success/len(df_feat)*100:.0f}%)')
    
    return df_feat

# Run
feature_dfs = {}
for approach, df in datasets.items():
    feature_dfs[approach] = extract_all_features(df, approach, gdd_data)


## 5. Feature Summary

In [ ]:
for approach, df in feature_dfs.items():
    print(f'\n{"="*60}')
    print(f'{approach.upper()} Features')
    print(f'{"="*60}')
    
    # Separate target and meta from features
    meta_cols = ['field_id', 'year', 'flag_true_doy', 'n_obs']
    feat_cols = [c for c in df.columns if c not in meta_cols]
    
    print(f'Samples: {len(df)}')
    print(f'Feature columns: {len(feat_cols)}')
    print(f'Target: flag_true_doy')
    
    print(f'\nFeature categories:')
    for vi in VIS:
        vi_feats = [c for c in feat_cols if c.startswith(vi)]
        print(f'  {vi}: {len(vi_feats)} features — {vi_feats}')
    dl_feats = [c for c in feat_cols if c.startswith('DL_')]
    print(f'  Double Logistic: {len(dl_feats)} features — {dl_feats}')
    env_feats = [c for c in feat_cols if c in ['latitude', 'longitude', 'photoperiod_at_SOS', 'GDD_at_SOS', 'GDD_M2_at_SOS', 'VD_at_SOS', 'fV_at_SOS', 'GDD_eff_at_SOS', 'PPT_at_SOS', 'PPT_greenup']]
    print(f'  Environmental: {len(env_feats)} features — {env_feats}')
    
    print(f'\nMissing values per feature:')
    for col in feat_cols:
        n_miss = df[col].isna().sum()
        if n_miss > 0:
            print(f'  {col}: {n_miss} ({n_miss/len(df)*100:.1f}%)')
    
    # Save
    out_path = os.path.join(OUTPUT_DIR, f'features_{approach}.parquet')
    df.to_parquet(out_path, index=False)
    print(f'\nSaved to: {out_path}')